# Assignment — vessel PINN

Builds on `notebooks/pinn_vessel_ode.ipynb`. Solution cells use `### BEGIN/END SOLUTION` (nbgrader).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..','..')))
import numpy as np, torch, matplotlib.pyplot as plt
from pyutils import pinn_core as pc
from pyutils.pinn_core import MLP, grad, train
from pyutils.benchmarks import vessel
torch.set_default_dtype(torch.float64); torch.manual_seed(0); np.random.seed(0)
pc.set_plot_style()


## Task — recover BOTH damping coefficients (and meet identifiability)

From a noisy surge coast-down, recover **both** the linear damping $d_u$ and the quadratic damping $X_{u|u|}$ jointly. 

Expect the **linear** term to be well recovered but the **quadratic** term to be only loosely constrained from a single low-speed coast-down — this is the *persistency-of-excitation* lesson: a maneuver that does not strongly excite a term cannot identify it well. (Try adding a second coast-down from a higher $u_0$ to improve it.)

In [2]:
p=vessel.VesselParams(); m11=p.m-p.X_udot; u0,T=2.0,20.0
t_gt,Z=vessel.simulate(lambda t,z: np.zeros(3), z0=[0,0,0,u0,0,0], t_end=T, dt=0.01, p=p)
u_gt=Z[:,3]
du_hat=Xuu_hat=None
### BEGIN SOLUTION
net=MLP([1,32,32,1])
log_du=torch.tensor(np.log(30.0),requires_grad=True)
log_X =torch.tensor(np.log(20.0),requires_grad=True)
nd=50; idx=np.linspace(0,len(t_gt)-1,nd).astype(int)
td=torch.tensor(t_gt[idx]).reshape(-1,1); ud=torch.tensor(u_gt[idx]+0.02*np.random.randn(nd)).reshape(-1,1)
u=lambda t: u0+t*net(t/T)
def loss():
    du=torch.exp(log_du); X=torch.exp(log_X)
    t=(torch.rand(2000,1)*T).requires_grad_(True); uu=u(t)
    Lr=((m11*grad(uu,t)+du*uu+X*torch.abs(uu)*uu)**2).mean()
    return Lr+100*((u(td)-ud)**2).mean()
train(loss, list(net.parameters())+[log_du,log_X], steps=6000, lr=2e-3)
du_hat=float(torch.exp(log_du).detach()); Xuu_hat=float(torch.exp(log_X).detach())
### END SOLUTION
print(f'd_u={du_hat:.2f} (true {p.Du}),  Xuu={Xuu_hat:.2f} (true {p.Xuu})')
# linear damping well identified; quadratic only loosely (identifiability)
assert abs(du_hat-p.Du)/p.Du < 0.25
assert abs(Xuu_hat-p.Xuu)/p.Xuu < 0.6

d_u=11.04 (true 10.0),  Xuu=7.47 (true 5.0)
